# Dask ETL: Distributed Raster Reprojection

Reproject a subset of the USGS 10m seamless DEM from NAD83 geographic coordinates to a projected coordinate system using Dask.

### Overview

This notebook reads elevation data from a 29 TB Zarr store, clips it to a region of interest, and reprojects it from EPSG:4269 (NAD83 lat/lon) to EPSG:5070 (NAD83 / Conus Albers Equal Area). The full dataset is far too large to fit in memory, so every step stays lazy until the final write. The Dask dashboard lets you watch the reprojection work move through the cluster in real time.

Imports

In [ ]:
import numpy as np
import xarray as xr
import dask
import dask.array as da
import matplotlib.pyplot as plt
from dask.distributed import Client, LocalCluster
from pathlib import Path

from xrspatial import reproject

## Cluster Setup

In [ ]:
cluster = LocalCluster(
    n_workers=20,
    threads_per_worker=1,
    memory_limit="2GB",
)
client = Client(cluster)
client

The dashboard is at [http://localhost:8787/status](http://localhost:8787/status) by default. It shows task progress and memory pressure per worker. Adjust `n_workers` and `memory_limit` to match your machine. Four workers at 2 GB each works well on a 16 GB system.

## Extract

The source data is a USGS 10m seamless DEM stored as a chunked Zarr archive. The full extent is roughly 940k by 3.9M pixels at ~10 meter resolution. We'll open it lazily and clip to a smaller region.

In [ ]:
ZARR_PATH = Path.home() / "elevation" / "usgs10m_dem_c6.zarr"

ds = xr.open_zarr(ZARR_PATH)
ds

In [ ]:
ds.xrs.preview().plot()

Nothing has been read from disk yet. The repr above shows the Dask task graph backing the array. Each chunk is 2048 x 2048 pixels.

Let's clip to Colorado. Good mix of flat plains and mountains, and small enough to finish in a reasonable time.

In [ ]:
# Bounding box: roughly Colorado plus some margin
lat_min, lat_max = 32.5, 45.5
lon_min, lon_max = -113.5, -95.5

dem = ds["usgs10m_dem"].sel(
    y=slice(lat_max, lat_min),
    x=slice(lon_min, lon_max),
)
dem

In [ ]:
print(f"Subset shape: {dem.shape}")
print(f"Subset size:  {dem.nbytes / 1e9:.1f} GB")
print(f"Chunk layout: {dem.chunks}")

Even this subset is several GB. The chunked layout means Dask can process it in parallel without loading the whole thing.

In [ ]:
dem.xrs.preview().plot()

## Transform

The source CRS is EPSG:4269 (NAD83, geographic lat/lon). We'll reproject to EPSG:5070 (NAD83 / Conus Albers Equal Area Conic), which gives equal-area cells in meters. That matters any time pixel area feeds into a calculation, like drainage area or cut/fill volumes, and it makes the DEM compatible with other projected datasets.

`xrspatial.reproject` handles Dask arrays natively. It builds a lazy task graph where each output chunk is reprojected independently using numba-JIT'd resampling kernels.

In [ ]:
SOURCE_CRS = "EPSG:4269"
TARGET_CRS = "EPSG:5070"
TARGET_RES = 10  # 10 meter output pixels

dem_projected = reproject(
    dem,
    TARGET_CRS,
    source_crs=SOURCE_CRS,
    resolution=TARGET_RES,
    resampling="nearest",
    nodata=np.nan,
    chunk_size=2048,
)
dem_projected

The result is still lazy. The repr shows the projected coordinate arrays and the new shape. No pixels have been resampled yet.

In [ ]:
print(f"Output shape: {dem_projected.shape}")
print(f"Output size:  {dem_projected.nbytes / 1e9:.1f} GB")
print(f"Output CRS:   {dem_projected.attrs.get('crs', 'not set')[:60]}...")

## Load

Write the reprojected DEM to a new Zarr store. Zarr supports parallel writes natively, so each Dask worker can write its chunks independently. This is where all the actual computation happens.

In [ ]:
OUTPUT_PATH = Path("colorado_dem_5070.zarr")

# Package as a Dataset for to_zarr
ds_out = dem_projected.to_dataset(name="elevation")

ds_out.to_zarr(
    OUTPUT_PATH,
    mode="w",
    consolidated=True,
)

The write triggers the full computation. Watch the dashboard to see tasks moving through the cluster.

In [ ]:
# Verify the output
ds_check = xr.open_zarr(OUTPUT_PATH)
print(ds_check)
print(f"\nOutput CRS: {ds_check['elevation'].attrs.get('crs', 'not set')[:60]}...")
print(f"Shape:      {ds_check['elevation'].shape}")
print(f"Size:       {ds_check['elevation'].nbytes / 1e9:.1f} GB")

In [ ]:
ds_check.xrs.preview().plot()

## Cleanup

In [ ]:
client.close()
cluster.close()

### What's next

The whole pipeline stayed lazy until the Zarr write, so memory usage stayed within the 2 GB per-worker limit even though the input was several GB.

From here you could:

- Expand the bounding box (or use the full CONUS extent) and add more workers
- Switch to bilinear or cubic resampling (`resampling='bilinear'` or `'cubic'`) for smoother output
- Chain xarray-spatial transforms (slope, hillshade) before the write
- Output to Cloud-Optimized GeoTIFF instead of Zarr if you need HTTP range-request access